<a href="https://colab.research.google.com/github/yubou0/LTC_policy3.0_RAG/blob/main/LTC_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#長照 3.0 政策諮詢 RAG 系統

## 1.基本資訊
- **專案動機**:「長照十年計畫 3.0核定本」長達 181 頁，內容包含大量法律術語與行政細節，對一般民眾或照護者而言閱讀門檻極高。本專案透過 RAG (Retrieval-Augmented Generation) 技術，建立一個具備事實根據的問答系統，讓使用者能透過自然語言快速檢索核心政策。同時自我練習使用RAG
- **Data Sources**:衛生福利部官方發布之《長照十年計畫 3.0 (115-124 年) - 核定本》PDF 文件


## 2.requirements
Python 3.12.12

langchain

langchain-community

langchain-classic

langchain-google-genai

langchain-huggingface

pypdf

chromadb

sentence-transformers

## 安裝RAG所需的關鍵套件

In [ ]:
!pip install -q -U langchain langchain-community langchain-classic langchain-google-genai langchain-huggingface pypdf chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.

## 讀取與切割文件
使用 PyPDFLoader 讀取 181 頁的政策文件，並使用 RecursiveCharacterTextSplitter 將文本切分為 800 字的小片段，並設置 100 字的重疊區間以確保語意不中斷。

In [ ]:
import sys
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 抓PDF (路徑在 /content/)
file_path = "/content/長照十年計畫3.0(115-124年)-核定本.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()

# chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
split_docs = text_splitter.split_documents(pages)

print(f"success！total pages：{len(pages)}")
print(f"success！chunks：{len(split_docs)}")

success！total pages：182
success！chunks：246


## 向量化&建立vectorDB
採用 paraphrase-multilingual-MiniLM-L12-v2 嵌入模型。這是一個針對多國語言優化的輕量化模型，並選用 Colab 的 T4 GPU 進行。

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 自動偵測是否可以使用 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"正在使用設備: {device}")

# 初始化Embedding模型
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': device}
)

# 建立VectorDB(存在 Colab 暫存目錄)
vector_db = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print("success set vectorDB")

正在使用設備: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

success set vectorDB


## RAG Chain Construction
採用 LangChain 最新版之 create_retrieval_chain 架構，連結 Gemini 1.5 Flash API。透過 System Prompt 的設計，嚴格限制 AI 只能根據檢索到的「文件片段」回答，以避免幻覺

參考資料:(https://github.com/langchain-ai/langchain/issues/33822)

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# LangChain v1.0
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. API Key
os.environ["GOOGLE_API_KEY"] = "your_api_key"

# 2. Gemini
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 3. Prompt
system_prompt = (
    "你只會回答文件中有提及的內容。若是文件沒提到，則回覆文件未提及。請根據以下內容回答問題。"
    "內容：{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 使用從 langchain_classic 導入的函數
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(vector_db.as_retriever(), combine_docs_chain)

## 實際QA

In [ ]:
# 5. 測試
query = "醫照整合的具體做法有哪些？"
response = rag_chain.invoke({"input": query}) #提問轉成向量，去DB找接近的

print(f"答：\n{response['answer']}")

答：
根據文件內容，醫照整合的具體做法包括：

*   **建立在宅責任醫師照顧網絡：**
    *   推動在宅責任醫師制度：由家庭醫師提供以人為中心的整合性健康管理模式，包含預防保健、健康促進、疾病管理、在宅醫療等照護服務，並結合醫院與社區醫療團隊及早發現潛在長照需求者，強化在宅/照護機構之急症照護及安寧療護，串連社區生活照顧協同單位，協助連結所需醫療照護服務及長照服務資源。
    *   建構住宿型照護機構住民整合照護模式：由醫療團隊進駐住宿型照護機構，採跨專業合作機制共同照護機構住民，整合醫療、照顧及復能、心理輔導等其他服務，並以論人計酬之支付概念，促使照護機構主動導入預防保健、減少機構內感染，建立每位住民個別化之照護措施。
    *   建置智慧化在宅醫療生態系：利用數位科技整合醫療照護。
*   **加速長照服務銜接時效：**
    *   衛福部規劃由醫院出備評估小組，於個案出院日前完成長照需要等級評估，並至照管系統擬定簡易個案照顧計畫後，聯絡長照個案居住地之各縣市照管中心。
*   **落實照護機構專責醫療機構機制：**
    *   衛福部透過獎勵計畫落實照護機構專責醫療機構機制，減少住民外出就醫，降低住民及陪同就醫人員往返醫療機構之感染風險，並藉由醫療機構之專責管理，掌握住民之健康情形及控制慢性病之惡化。
    *   長照 3.0 將持續整合長照「減少住宿型機構住民至醫療機構就醫方案」及健保巡診醫療機構。


# mertice
若是想量化結果，可使用RAGAS去做評估

Ragas 採用 LLM-as-a-Judge 的技術，利用更高階模型具備的推理能力對模型進行驗證

關鍵指標

- Faithfulness,驗證回答是否嚴格遵守 PDF 原文，確保政策不被誤傳。

- Answer Relevancy,確保系統能精確回答提問，不產生冗餘資訊。

- Context Precision,驗證 Embedding 模型是否能定位核心條文。

## 長照 3.0 政策諮詢 RAG 系統概述

本 Colab Notebook 旨在建立一個基於 Retrieval-Augmented Generation (RAG) 技術的智能問答系統，用於提供關於「長照十年計畫 3.0 核定本」政策的諮詢服務。由於原始政策文件內容龐大且專業術語繁多，本系統透過自然語言處理技術，讓使用者能夠快速、準確地檢索並理解政策核心內容。

**核心流程包括：**

1.  **文件讀取與切割**：利用 `PyPDFLoader` 載入 PDF 政策文件，並使用 `RecursiveCharacterTextSplitter` 將文件切分成易於檢索的小片段。
2.  **向量化與建立向量資料庫 (VectorDB)**：採用多語言優化的 `paraphrase-multilingual-MiniLM-L12-v2` 嵌入模型，將文本片段轉換為向量表示，並儲存於 `Chroma` 向量資料庫中，以便進行高效的語義搜索。
3.  **RAG Chain 建構**：整合 LangChain 的 `create_retrieval_chain` 與 Google Gemini 1.5 Flash API。透過精心設計的 System Prompt，限制 AI 模型的回答僅基於檢索到的文件內容，有效避免「幻覺」現象。
4.  **實際問答與驗證**：系統能夠接受使用者提問，從向量資料庫中檢索相關文本片段，並結合大型語言模型生成具事實基礎的回答。同時，也初步提到了使用 RAGAS 評估系統性能的重要性，關注回答的忠實性 (Faithfulness)、相關性 (Answer Relevancy) 和上下文精確度 (Context Precision)。

透過這個系統，希望能降低民眾理解長照政策的門檻，提升資訊獲取的效率與準確性。